<a href="https://colab.research.google.com/github/norah30/bookspace/blob/main/Integracion_Predictivo_MongoDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Integración de Análisis Predictivo en MongoDB (NoSQL)
### Pronóstico deportivo internacional · KNN + Regresión Lineal → MongoDB Atlas desde Google Colab

**Demo / práctica.** Este notebook consolida los resultados de dos modelos sobre un dataset
de resultados de fútbol internacional (1872–2026, incluye Mundiales) y los carga en **MongoDB Atlas**.

| Trabajo previo | Modelo | Tarea | Salida |
|---|---|---|---|
| **A Autónomo 1** | KNN | Clasificación | Resultado del partido: `Local` / `Empate` / `Visitante` |
| **A Autónomo 2** | Regresión Lineal | Regresión | Total de goles del partido (valor predicho, R², error) |

**Dataset:** `martj42/international_results` (repositorio público en GitHub; también está en Kaggle).
Se descarga directo por URL para que el notebook sea reproducible en Colab sin subir archivos.


## ⚠️ Importante: conectarse a MongoDB desde Google Colab

El ejemplo típico `MongoClient("mongodb://localhost:27017")` **NO funciona en Colab**: Colab corre
en la nube de Google y no puede ver un MongoDB instalado en tu PC local. Para esto usamos **MongoDB Atlas**
(Mongo gestionado en la nube) con un string `mongodb+srv://...`.

Cosas que el ejemplo del enunciado no menciona y que **rompen** la conexión si las omites:

1. **`pip install pymongo` no basta.** El esquema `mongodb+srv://` necesita resolución DNS SRV →
   hay que instalar también **`dnspython`** (o `pymongo[srv]`).
2. **Network Access en Atlas:** agrega `0.0.0.0/0` (o la IP de Colab) en *Atlas → Network Access*,
   o la conexión quedará colgada por timeout.
3. **Password con caracteres especiales** (`@`, `:`, `/`, `#`) → hay que URL-encodearlo en el string.
4. **Credenciales fuera del código:** aquí usamos `getpass` (o *Colab Secrets*) para no dejar el
   usuario/clave escritos en una celda.
5. **Tipos `numpy`** (`int64`, `float64`) → `pymongo` real a veces los rechaza. Saneamos a tipos
   nativos de Python antes de insertar.
6. Si aparece un error SSL/TLS, pasamos `tlsCAFile=certifi.where()`.


In [ ]:
# === Instalación de dependencias (en Colab) ===
# pymongo[srv] incluye dnspython, necesario para los strings mongodb+srv:// de Atlas
!pip install -q "pymongo[srv]" dnspython certifi scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.6 MB/s eta 0:00:00


---
## Parte 1 · Preparación del dataset consolidado

Pasos: (1) cargar y limpiar, (2) construir variables predictoras a partir del historial de cada
selección, (3) definir los objetivos, (4) entrenar KNN y Regresión, (5) consolidar todo en un único
DataFrame con nombres semánticos y **sin nulos**.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error, mean_absolute_error

RANDOM_STATE = 42

# --- Carga directa desde repositorio público (GitHub raw) ---
URL = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
df = pd.read_csv(URL)
df["date"] = pd.to_datetime(df["date"])

# Limpieza: quitar partidos no jugados (sin marcador) y quedarnos con la era moderna
df = df.dropna(subset=["home_score", "away_score"])
df = df[df["date"] >= "2000-01-01"].reset_index(drop=True)
df["home_score"] = df["home_score"].astype(int)
df["away_score"] = df["away_score"].astype(int)

print(f"Partidos jugados desde 2000: {len(df):,}")
df.head(3)

Partidos jugados desde 2000: 25,391


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,2000-01-04,Egypt,Togo,2,1,Friendly,Aswan,Egypt,False
1,2000-01-07,Tunisia,Togo,7,0,Friendly,Tunis,Tunisia,False
2,2000-01-08,Trinidad and Tobago,Canada,0,0,Friendly,Port of Spain,Trinidad and Tobago,False


In [ ]:
# --- Variables predictoras: historial de cada selección ---
# Tabla "larga": cada partido aporta una fila por equipo (local y visitante)
home = df[["home_team","home_score","away_score"]].rename(
    columns={"home_team":"team","home_score":"gf","away_score":"ga"})
away = df[["away_team","away_score","home_score"]].rename(
    columns={"away_team":"team","away_score":"gf","home_score":"ga"})
largo = pd.concat([home, away], ignore_index=True)
largo["win"] = (largo["gf"] > largo["ga"]).astype(int)

# Promedios históricos por selección (goles a favor, en contra, % de victorias)
stats = largo.groupby("team").agg(
    prom_gf=("gf","mean"),
    prom_gc=("ga","mean"),
    winrate=("win","mean"),
    pj=("gf","size")).reset_index()
stats = stats[stats["pj"] >= 10]   # solo selecciones con historial mínimo

# Unimos las stats como features del local y del visitante
df = df.merge(stats.add_prefix("loc_"), left_on="home_team", right_on="loc_team", how="inner")
df = df.merge(stats.add_prefix("vis_"), left_on="away_team", right_on="vis_team", how="inner")
df["es_mundial"] = (df["tournament"] == "FIFA World Cup").astype(int)
df["neutral"]    = df["neutral"].astype(int)
print("Features listas. Partidos con historial suficiente:", len(df))

Features listas. Partidos con historial suficiente: 25176


> **Nota didáctica (sin esconder el polvo bajo la alfombra):** aquí las medias por selección
> se calculan sobre todo el período por simplicidad. En **producción** se calcularían usando solo
> partidos *anteriores* a cada fecha (features tipo *rolling*) para evitar fuga de información (*data leakage*).

In [ ]:
# --- Objetivos de los dos modelos ---
def resultado(r):
    if r.home_score > r.away_score: return "Local"
    if r.home_score < r.away_score: return "Visitante"
    return "Empate"

df["resultado_real"]   = df.apply(resultado, axis=1)          # objetivo del KNN
df["total_goles_real"] = df["home_score"] + df["away_score"]  # objetivo de la regresión

FEATURES = ["loc_prom_gf","loc_prom_gc","loc_winrate",
            "vis_prom_gf","vis_prom_gc","vis_winrate",
            "neutral","es_mundial"]

X = df[FEATURES].values
idx = np.arange(len(df))
i_tr, i_te = train_test_split(idx, test_size=0.25, random_state=RANDOM_STATE,
                              stratify=df["resultado_real"])
print(f"Entrenamiento: {len(i_tr):,}  |  Prueba: {len(i_te):,}")

Entrenamiento: 18,882  |  Prueba: 6,294


In [ ]:
# --- A Autónomo 1 · KNN (clasificación) ---
scaler = StandardScaler().fit(X[i_tr])      # KNN es sensible a escala -> estandarizar
Xs = scaler.transform(X)

knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(Xs[i_tr], df["resultado_real"].values[i_tr])
pred_knn = knn.predict(Xs[i_te])

acc = accuracy_score(df["resultado_real"].values[i_te], pred_knn)
print(f"KNN  ->  accuracy (test): {acc:.3f}   (azar de 3 clases = 0.333)")

KNN  ->  accuracy (test): 0.558   (azar de 3 clases = 0.333)


In [ ]:
# --- A Autónomo 2 · Regresión Lineal (total de goles) ---
y = df["total_goles_real"].values
reg = LinearRegression().fit(X[i_tr], y[i_tr])
pred_reg = reg.predict(X[i_te])

r2   = r2_score(y[i_te], pred_reg)
rmse = np.sqrt(mean_squared_error(y[i_te], pred_reg))
mae  = mean_absolute_error(y[i_te], pred_reg)
print(f"Regresión  ->  R2={r2:.3f}   RMSE={rmse:.3f}   MAE={mae:.3f}")

Regresión  ->  R2=0.107   RMSE=1.867   MAE=1.422


> **Sobre el R² bajo (~0.11):** es **esperado y honesto**. Predecir el número exacto de goles
> de un partido es de altísima varianza (un penal, una expulsión, un rebote lo cambian todo).
> El objetivo de esta práctica **no** es maximizar el R², sino el *pipeline* de integración:
> consolidar → cargar en NoSQL → consultar. Aun así, el RMSE de ~1.9 goles es un punto de partida razonable.

In [ ]:
# --- DataFrame consolidado (solo set de prueba: predicciones out-of-sample) ---
final = df.iloc[i_te].copy()
final["prediccion_KNN"]        = pred_knn
final["regresion_goles_pred"]  = np.round(pred_reg, 3)
final["reg_error_abs"]         = np.round(np.abs(y[i_te] - pred_reg), 3)  # error por partido
final["reg_r2_modelo"]         = round(float(r2), 4)    # métrica global del modelo
final["reg_rmse_modelo"]       = round(float(rmse), 4)

# Renombrado semántico (claridad)
ren = {"date":"fecha","home_team":"equipo_local","away_team":"equipo_visitante",
       "home_score":"goles_local","away_score":"goles_visitante","tournament":"torneo",
       "city":"ciudad","country":"pais_sede","neutral":"cancha_neutral",
       "loc_prom_gf":"local_prom_gf","loc_prom_gc":"local_prom_gc","loc_winrate":"local_winrate",
       "vis_prom_gf":"visit_prom_gf","vis_prom_gc":"visit_prom_gc","vis_winrate":"visit_winrate"}
final = final.rename(columns=ren)

keep = list(ren.values()) + ["es_mundial","resultado_real","total_goles_real",
        "prediccion_KNN","regresion_goles_pred","reg_error_abs","reg_r2_modelo","reg_rmse_modelo"]
final = final[keep].copy()
final["fecha"] = final["fecha"].dt.strftime("%Y-%m-%d")
final["anio"]  = final["fecha"].str[:4].astype(int)
for c in ["local_prom_gf","local_prom_gc","local_winrate","visit_prom_gf","visit_prom_gc","visit_winrate"]:
    final[c] = final[c].round(3)

print("Forma del DataFrame final:", final.shape)
print("Valores nulos totales:", int(final.isnull().sum().sum()))   # debe ser 0
final.head()

Forma del DataFrame final: (6294, 24)
Valores nulos totales: 0


,fecha,equipo_local,equipo_visitante,goles_local,goles_visitante,torneo,ciudad,pais_sede,cancha_neutral,local_prom_gf,...,visit_winrate,es_mundial,resultado_real,total_goles_real,prediccion_KNN,regresion_goles_pred,reg_error_abs,reg_r2_modelo,reg_rmse_modelo,anio
18025,2019-01-13,Turkmenistan,Uzbekistan,0,4,AFC Asian Cup,Dubai,United Arab Emirates,1,1.504,...,0.466,0,Visitante,4,Visitante,2.706,1.294,0.1068,1.8667,2019
1718,2001-07-13,Isle of Man,Shetland,2,2,Island Games,Castletown,Isle of Man,0,3.467,...,0.378,0,Empate,4,Local,4.390,0.390,0.1068,1.8667,2001
21455,2022-11-20,South Africa,Angola,1,1,Friendly,Mbombela,South Africa,0,1.261,...,0.338,0,Empate,2,Local,2.026,0.026,0.1068,1.8667,2022
24499,2025-10-12,Djibouti,Sierra Leone,1,2,FIFA World Cup qualification,Casablanca,Morocco,1,0.602,...,0.263,0,Visitante,3,Visitante,2.873,0.127,0.1068,1.8667,2025
13965,2014-10-10,Chile,Peru,3,0,Friendly,Valparaíso,Chile,0,1.332,...,0.328,0,Local,3,Local,2.426,0.574,0.1068,1.8667,2014


---
## Parte 2 · Transformación a JSON y carga en MongoDB Atlas

In [ ]:
# --- DataFrame -> lista de diccionarios + saneo de tipos numpy ---
def limpiar(v):
    if isinstance(v, np.integer): return int(v)
    if isinstance(v, np.floating): return float(v)
    if isinstance(v, np.bool_):    return bool(v)
    return v

registros = [{k: limpiar(v) for k, v in r.items()} for r in final.to_dict(orient="records")]
print("Registros listos para insertar:", len(registros))
registros[0]

Registros listos para insertar: 6294


{'fecha': '2019-01-13',
 'equipo_local': 'Turkmenistan',
 'equipo_visitante': 'Uzbekistan',
 'goles_local': 0,
 'goles_visitante': 4,
 'torneo': 'AFC Asian Cup',
 'ciudad': 'Dubai',
 'pais_sede': 'United Arab Emirates',
 'cancha_neutral': 1,
 'local_prom_gf': 1.504,
 'local_prom_gc': 1.504,
 'local_winrate': 0.382,
 'visit_prom_gf': 1.585,
 'visit_prom_gc': 1.095,
 'visit_winrate': 0.466,
 'es_mundial': 0,
 'resultado_real': 'Visitante',
 'total_goles_real': 4,
 'prediccion_KNN': 'Visitante',
 'regresion_goles_pred': 2.706,
 'reg_error_abs': 1.294,
 'reg_r2_modelo': 0.1068,
 'reg_rmse_modelo': 1.8667,
 'anio': 2019}

In [ ]:
# --- Conexión a MongoDB Atlas ---
import certifi
from urllib.parse import quote_plus
from pymongo import MongoClient

# Datos del cluster. El HOST lo da Atlas en: Connect -> Drivers (parte después de la '@').
MONGO_USER = "alumno_uide"
MONGO_PASS = "alumno_uide"             # cuenta de clase compartida; para algo real, usar getpass/Secrets
MONGO_HOST = "cluster0.hyft2gh.mongodb.net"  # host de tu cluster (de tu connection string)

# Alternativa segura (sin clave escrita en el código):
# from getpass import getpass
# MONGO_PASS = getpass("Password de Atlas: ")

# quote_plus escapa caracteres especiales del usuario/clave en el string
MONGO_URI = (f"mongodb+srv://{quote_plus(MONGO_USER)}:{quote_plus(MONGO_PASS)}"
             f"@{MONGO_HOST}/?retryWrites=true&w=majority&appName=Cluster0")

client = MongoClient(MONGO_URI, tlsCAFile=certifi.where(), serverSelectionTimeoutMS=20000)
client.admin.command("ping")          # falla rápido si el host, las credenciales o la IP están mal
print("✅ Conexión a Atlas correcta")

db = client["proyecto_bigdata"]
coleccion = db["resultados_modelos"]

✅ Conexión a Atlas correcta


In [ ]:
# --- Inserción + verificación de que cargó EN LÍNEA ---
coleccion.delete_many({})                 # idempotente: limpia antes de recargar
n_ins = len(coleccion.insert_many(registros).inserted_ids)

# Releer desde Atlas para CONFIRMAR la carga real (no solo confiar en el insert)
n_base  = coleccion.count_documents({})
muestra = coleccion.find_one({}, {"_id":0,"equipo_local":1,"equipo_visitante":1,"prediccion_KNN":1})

print(f"Insertados: {n_ins:,}   |   Releídos desde Atlas: {n_base:,}")
print("Muestra leída de la base:", muestra)
print("CARGA EN LÍNEA:", "✅ OK" if n_ins == n_base == len(registros) else "❌ revisar")

Insertados: 6,294   |   Releídos desde Atlas: 6,294
Muestra leída de la base: {'equipo_local': 'Turkmenistan', 'equipo_visitante': 'Uzbekistan', 'prediccion_KNN': 'Visitante'}
CARGA EN LÍNEA: ✅ OK


---
## Parte 3 · Consultas en MongoDB

Cinco consultas que combinan `count_documents`, `find` y un *pipeline* de agregación.

In [ ]:
# [1] Total de registros cargados
total = coleccion.count_documents({})
print("[1] Total de documentos en la colección:", total)

[1] Total de documentos en la colección: 6294


In [ ]:
# [2] Filtrado por una predicción específica del KNN
n_local = coleccion.count_documents({"prediccion_KNN": "Local"})
print(f"[2] Partidos donde el KNN predice 'Local': {n_local}")

# Tres ejemplos concretos (proyectando solo algunos campos)
proj = {"_id":0, "equipo_local":1, "equipo_visitante":1, "torneo":1, "prediccion_KNN":1}
for d in coleccion.find({"prediccion_KNN": "Local"}, proj).limit(3):
    print("   ", d)

[2] Partidos donde el KNN predice 'Local': 3686
    {'equipo_local': 'Isle of Man', 'equipo_visitante': 'Shetland', 'torneo': 'Island Games', 'prediccion_KNN': 'Local'}
    {'equipo_local': 'South Africa', 'equipo_visitante': 'Angola', 'torneo': 'Friendly', 'prediccion_KNN': 'Local'}
    {'equipo_local': 'Chile', 'equipo_visitante': 'Peru', 'torneo': 'Friendly', 'prediccion_KNN': 'Local'}


In [ ]:
# [3] Búsqueda por umbral de error en la regresión
UMBRAL = 3.0   # goles de error absoluto
n_err = coleccion.count_documents({"reg_error_abs": {"$gt": UMBRAL}})
print(f"[3] Partidos con error de regresión > {UMBRAL} goles: {n_err}  ({n_err/total:.1%} del total)")

proj = {"_id":0,"equipo_local":1,"equipo_visitante":1,"total_goles_real":1,
        "regresion_goles_pred":1,"reg_error_abs":1}
for d in coleccion.find({"reg_error_abs": {"$gt": UMBRAL}}, proj).sort("reg_error_abs",-1).limit(3):
    print("   ", d)

[3] Partidos con error de regresión > 3.0 goles: 480  (7.6% del total)
    {'equipo_local': 'Australia', 'equipo_visitante': 'American Samoa', 'total_goles_real': 31, 'regresion_goles_pred': 7.527, 'reg_error_abs': 23.473}
    {'equipo_local': 'Guam', 'equipo_visitante': 'North Korea', 'total_goles_real': 21, 'regresion_goles_pred': 3.494, 'reg_error_abs': 17.506}
    {'equipo_local': 'South Korea', 'equipo_visitante': 'Nepal', 'total_goles_real': 16, 'regresion_goles_pred': 2.982, 'reg_error_abs': 13.018}


In [ ]:
# [4] Filtro temático: solo partidos de Copa del Mundo
n_mundial = coleccion.count_documents({"torneo": "FIFA World Cup"})
print(f"[4] Partidos de FIFA World Cup en la colección: {n_mundial}")

[4] Partidos de FIFA World Cup en la colección: 108


In [ ]:
# [5] Agregación: error medio de la regresión por clase predicha del KNN
pipeline = [
    {"$group": {"_id": "$prediccion_KNN",
                "n": {"$sum": 1},
                "error_medio_goles": {"$avg": "$reg_error_abs"}}},
    {"$sort": {"n": -1}}
]
print("[5] Error medio de la regresión según la clase que predijo el KNN:")
for d in coleccion.aggregate(pipeline):
    print(f"    {d['_id']:<10}  n={d['n']:<5}  error_medio={d['error_medio_goles']:.3f} goles")

[5] Error medio de la regresión según la clase que predijo el KNN:
    Local       n=3686   error_medio=1.459 goles
    Visitante   n=1673   error_medio=1.412 goles
    Empate      n=935    error_medio=1.294 goles


**Documentación de los resultados y su utilidad**

- **[1] Total:** confirma que la carga fue completa (control de integridad del *pipeline*).
- **[2] Filtro por predicción:** permite segmentar partidos por pronóstico del KNN; base directa para
  alimentar un panel ("¿qué partidos predice como victoria local?") o una API de recomendaciones.
- **[3] Umbral de error:** aísla los partidos donde el modelo de goles falló más → insumo para
  *monitoreo de calidad del modelo* y reentrenamiento dirigido.
- **[4] Filtro temático (Mundial):** muestra cómo NoSQL facilita *slices* por contexto sin re-procesar archivos.
- **[5] Agregación:** cruza ambos modelos en una sola consulta del lado de la base — algo costoso de
  hacer "a mano" sobre un CSV/Excel.

---
## Parte 4 · Preguntas de reflexión

**1. ¿Qué desafíos enfrentaste al integrar los datos en un formato adecuado para MongoDB?**
El principal fue de **tipos**: `pandas`/`numpy` usan `int64`, `float64` y `bool_`, que el driver
`pymongo` no siempre serializa; hubo que sanearlos a tipos nativos de Python. El segundo, propio de
**Colab → Atlas**: el `pip install pymongo` del enunciado no incluye `dnspython`, necesario para los
strings `mongodb+srv://`, y olvidar habilitar la IP en *Network Access* deja la conexión colgada.
Finalmente, decidir **qué guardar**: combinar variables originales, predicción categórica del KNN y
métricas de la regresión (valor predicho + error por fila + R²/RMSE globales) en un único documento coherente.

**2. ¿Qué ventajas percibes en NoSQL frente a CSV o Excel para análisis predictivo?**
*Esquema flexible:* cada documento puede tener campos distintos sin romper la colección, ideal cuando
los modelos evolucionan y agregan salidas nuevas. *Consultas declarativas* (`find`, agregaciones) sin
recargar todo el archivo en memoria. *Escalabilidad horizontal* (*sharding*) frente al límite práctico
de un Excel. *Acceso concurrente* multiusuario y vía red, mientras un CSV es un archivo estático que
alguien tiene que abrir. *JSON nativo*, que conecta sin fricción con APIs y dashboards web.

**3. ¿Qué considerar para escalar a producción?**
*Seguridad:* credenciales en variables de entorno/secret manager (nunca en el código), usuarios con
permisos mínimos, IP allowlist real en lugar de `0.0.0.0/0`. *Índices* sobre los campos más consultados
(`prediccion_KNN`, `reg_error_abs`, `torneo`) para que las consultas no hagan *full scan*. *Idempotencia
y versionado* de cargas (no duplicar al reejecutar). *Validación de esquema* (JSON Schema de Mongo).
*Reentrenamiento y monitoreo* del *drift* del modelo. *Inserciones por lotes* y manejo de errores/reintentos.
*Backups* y observabilidad.

**4. ¿Qué utilidad tendría esto en un dashboard, un sistema de recomendación o una API?**
*Dashboard:* las agregaciones (p. ej. error medio por clase, distribución de pronósticos) se grafican
directo; un BI se conecta a Mongo y refresca solo. *Recomendación:* filtrar partidos por predicción y
confianza para sugerir "value bets" o picks (caso de uso real de pronóstico deportivo). *API:* exponer
endpoints REST (`/predicciones?resultado=Local`) que sirven JSON listo desde la colección, sin
transformar nada en el camino.

**5. ¿Qué otros tipos de datos podrías almacenar y cómo complementarían el análisis?**
*JSON anidado:* metadatos del modelo (hiperparámetros, métricas, lista de features) en un subdocumento
— lo agregamos abajo como ejemplo. *Logs* de cada predicción (timestamp, versión del modelo, latencia)
para auditoría y *drift*. *Imágenes/binarios* (escudos, mapas de calor) vía GridFS. *Series temporales*
de odds o forma reciente. *Documentos de eventos* (alineaciones, lesiones). Todo ello enriquece el
contexto del pronóstico sin cambiar el motor de almacenamiento.

---
### Extra · Documento con JSON anidado (apoya la reflexión #5)
Guardamos los **metadatos del modelo** en una colección aparte para mostrar la flexibilidad de Mongo
con estructuras anidadas (algo imposible de representar limpio en un CSV plano).

In [ ]:
metadatos = {
    "proyecto": "pronostico_deportivo",
    "fecha_entrenamiento": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "dataset": {"fuente": "martj42/international_results", "n_partidos": int(len(df)),
                "desde": "2000-01-01"},
    "modelos": {
        "knn":       {"tipo": "KNeighborsClassifier", "k": 15,
                      "objetivo": "resultado_real", "accuracy": round(float(acc), 4)},
        "regresion": {"tipo": "LinearRegression", "objetivo": "total_goles_real",
                      "r2": round(float(r2), 4), "rmse": round(float(rmse), 4)}
    },
    "features": FEATURES
}
db["metadatos_modelos"].delete_many({"proyecto": "pronostico_deportivo"})
db["metadatos_modelos"].insert_one(metadatos)
print("Metadatos anidados insertados ✅")
db["metadatos_modelos"].find_one({}, {"_id": 0})

Metadatos anidados insertados ✅


{'proyecto': 'pronostico_deportivo',
 'fecha_entrenamiento': '2026-06-25 00:56',
 'dataset': {'fuente': 'martj42/international_results',
  'n_partidos': 25176,
  'desde': '2000-01-01'},
 'modelos': {'knn': {'tipo': 'KNeighborsClassifier',
   'k': 15,
   'objetivo': 'resultado_real',
   'accuracy': 0.5578},
  'regresion': {'tipo': 'LinearRegression',
   'objetivo': 'total_goles_real',
   'r2': 0.1068,
   'rmse': 1.8667}},
 'features': ['loc_prom_gf',
  'loc_prom_gc',
  'loc_winrate',
  'vis_prom_gf',
  'vis_prom_gc',
  'vis_winrate',
  'neutral',
  'es_mundial']}

---
## Parte 5 · Export opcional a `.json`

In [ ]:
import json

# Create a new list of dictionaries, excluding the '_id' field
registros_to_export = [{k: v for k, v in record.items() if k != '_id'} for record in registros]

with open("dataset_resultados_modelos.json", "w", encoding="utf-8") as f:
    json.dump(registros_to_export, f, ensure_ascii=False, indent=2)
print("Archivo JSON generado: dataset_resultados_modelos.json")

Archivo JSON generado: dataset_resultados_modelos.json


---
### Cierre
Pipeline completo y reproducible: **dataset público → KNN + Regresión → DataFrame consolidado sin nulos
→ MongoDB Atlas (desde Colab) → consultas y agregaciones → metadatos anidados → export JSON**.

Los puntos que diferencian "que funcione en Colab" de la receta del enunciado fueron: usar **Atlas** en
vez de `localhost`, instalar **`dnspython`**, habilitar **Network Access**, manejar **credenciales**
de forma segura y **sanear tipos numpy** antes de insertar.